# Function Transformer — Log Transform

Many models (especially linear/logistic regression) work best when features are roughly **normally distributed**. `Age` and `Fare` in the Titanic dataset are skewed — this notebook applies a **log transform** (`FunctionTransformer(np.log1p)`) to make them more symmetric, and checks whether that actually improves model accuracy.

## 1. Setup and Data

In [1]:
import pandas as pd
import numpy as np

import scipy.stats as stats

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

### ⚠️ Important: `fillna(..., inplace=True)` on a Column Is Broken on Current Pandas

`df['Age'].fillna(df['Age'].mean(), inplace=True)` is **chained assignment** — `df['Age']` returns a separate object first, and `inplace=True` then modifies *that*, not `df` itself. Older pandas let this slide (with a `FutureWarning`); as of **pandas 3.0** (Copy-on-Write is now always on), this raises `ChainedAssignmentError` outright, and even where it doesn't error, it silently leaves `df` completely unchanged. The fix is to call `fillna` on the DataFrame itself, targeting the column via a dict, or to reassign the column directly.

In [ ]:
df.fillna({'Age': df['Age'].mean()}, inplace=True)
# equivalent alternative: df['Age'] = df['Age'].fillna(df['Age'].mean())


In [ ]:
df['Age'].fillna(df['Age'].mean(),inplace=True)

In [ ]:
df.head()

In [ ]:
X = df.iloc[:,1:3]
y = df.iloc[:,0]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 2. Checking How Skewed the Raw Features Are

Two views side by side for each column: the **distribution shape** (left) and a **Q-Q plot** (right). A Q-Q plot compares a column's actual quantiles against what a perfectly normal distribution would produce — points hugging the diagonal line mean "close to normal"; a curve bending away from it (especially at the tails) reveals skewness.

### ⚠️ `sns.distplot()` Is Removed on Current Seaborn

Replaced with `sns.histplot(kde=True)`, as covered in the earlier visualization notebooks.

In [ ]:
plt.figure(figsize=(14, 4))
plt.subplot(121)
sns.histplot(X_train['Age'], kde=True)
plt.title('Age PDF')

plt.subplot(122)
stats.probplot(X_train['Age'], dist='norm', plot=plt)
plt.title('Age QQ Plot')

plt.show()


In [ ]:
plt.figure(figsize=(14, 4))
plt.subplot(121)
sns.histplot(X_train['Fare'], kde=True)
plt.title('Fare PDF')

plt.subplot(122)
stats.probplot(X_train['Fare'], dist='norm', plot=plt)
plt.title('Fare QQ Plot')

plt.show()


`Fare`'s Q-Q plot should bend away from the diagonal much more sharply than `Age`'s — it's a far more skewed column (a small number of very expensive tickets pull the distribution's tail out) — which is exactly the kind of column a log transform is meant to fix.

## 3. Baseline Accuracy — No Transform

In [ ]:
clf = LogisticRegression()
clf2 = DecisionTreeClassifier()

In [ ]:
clf.fit(X_train,y_train)
clf2.fit(X_train,y_train)
    
y_pred = clf.predict(X_test)
y_pred1 = clf2.predict(X_test)
    
print("Accuracy LR",accuracy_score(y_test,y_pred))
print("Accuracy DT",accuracy_score(y_test,y_pred1))

## 4. Applying the Log Transform to Both Columns

`np.log1p(x)` computes `log(1 + x)` rather than plain `log(x)` — the `+1` avoids `log(0)`, which is undefined (and `Fare` can be 0 for some passengers).

In [ ]:
trf = FunctionTransformer(func=np.log1p)

In [ ]:
X_train_transformed = trf.fit_transform(X_train)
X_test_transformed = trf.transform(X_test)

### Accuracy After Transforming Both Columns

In [ ]:
clf = LogisticRegression()
clf2 = DecisionTreeClassifier()

clf.fit(X_train_transformed,y_train)
clf2.fit(X_train_transformed,y_train)
    
y_pred = clf.predict(X_test_transformed)
y_pred1 = clf2.predict(X_test_transformed)
    
print("Accuracy LR",accuracy_score(y_test,y_pred))
print("Accuracy DT",accuracy_score(y_test,y_pred1))

### A More Reliable Comparison — Cross-Validation

A single `train_test_split()` accuracy can shift meaningfully depending on which rows land in the test set. Averaging accuracy across 10 different splits gives a steadier read on whether the transform is actually helping.

In [ ]:
X_transformed = trf.fit_transform(X)

clf = LogisticRegression()
clf2 = DecisionTreeClassifier()

print("LR",np.mean(cross_val_score(clf,X_transformed,y,scoring='accuracy',cv=10)))
print("DT",np.mean(cross_val_score(clf2,X_transformed,y,scoring='accuracy',cv=10)))

### Confirming the Effect Numerically — `.skew()`

The Q-Q plots below show the shape shift visually; `.skew()` puts a number on it — closer to 0 means more symmetric/normal. Comparing before vs. after confirms the log transform actually reduced skew, not just changed the plot's appearance.

In [ ]:
print('Fare skew  before:', X_train['Fare'].skew(), ' after:', X_train_transformed['Fare'].skew())
print('Age skew   before:', X_train['Age'].skew(),  ' after:', X_train_transformed['Age'].skew())


In [ ]:
plt.figure(figsize=(14,4))

plt.subplot(121)
stats.probplot(X_train['Fare'], dist="norm", plot=plt)
plt.title('Fare Before Log')

plt.subplot(122)
stats.probplot(X_train_transformed['Fare'], dist="norm", plot=plt)
plt.title('Fare After Log')

plt.show()

In [ ]:
plt.figure(figsize=(14,4))

plt.subplot(121)
stats.probplot(X_train['Age'], dist="norm", plot=plt)
plt.title('Age Before Log')

plt.subplot(122)
stats.probplot(X_train_transformed['Age'], dist="norm", plot=plt)
plt.title('Age After Log')

plt.show()

`Fare`'s Q-Q plot should look noticeably straighter after the log transform; `Age` — being much less skewed to begin with — likely shows only a small change, if any.

## 5. Transforming Only `Fare` — Since `Age` Didn't Need It

Given `Age` was already fairly close to normal, transforming it too may add no benefit (or even hurt slightly). `ColumnTransformer` applies the log transform to `Fare` only, leaving `Age` untouched via `remainder='passthrough'`.

In [ ]:
trf2 = ColumnTransformer([('log',FunctionTransformer(np.log1p),['Fare'])],remainder='passthrough')

X_train_transformed2 = trf2.fit_transform(X_train)
X_test_transformed2 = trf2.transform(X_test)

In [ ]:
clf = LogisticRegression()
clf2 = DecisionTreeClassifier()

clf.fit(X_train_transformed2,y_train)
clf2.fit(X_train_transformed2,y_train)
    
y_pred = clf.predict(X_test_transformed2)
y_pred2 = clf2.predict(X_test_transformed2)
    
print("Accuracy LR",accuracy_score(y_test,y_pred))
print("Accuracy DT",accuracy_score(y_test,y_pred2))

In [ ]:
X_transformed2 = trf2.fit_transform(X)

clf = LogisticRegression()
clf2 = DecisionTreeClassifier()

print("LR",np.mean(cross_val_score(clf,X_transformed2,y,scoring='accuracy',cv=10)))
print("DT",np.mean(cross_val_score(clf2,X_transformed2,y,scoring='accuracy',cv=10)))

## 6. Trying Other Transform Functions

`FunctionTransformer` accepts **any** function, not just `np.log1p` — this helper generalizes the fit → cross-validate → plot workflow so different transforms can be tried quickly on `Fare`.

In [ ]:
def apply_transform(transform):
    X = df.iloc[:,1:3]
    y = df.iloc[:,0]
    
    trf = ColumnTransformer([('log',FunctionTransformer(transform),['Fare'])],remainder='passthrough')
    
    X_trans = trf.fit_transform(X)
    
    clf = LogisticRegression()
    
    print("Accuracy",np.mean(cross_val_score(clf,X_trans,y,scoring='accuracy',cv=10)))
    
    plt.figure(figsize=(14,4))

    plt.subplot(121)
    stats.probplot(X['Fare'], dist="norm", plot=plt)
    plt.title('Fare Before Transform')

    plt.subplot(122)
    stats.probplot(X_trans[:,0], dist="norm", plot=plt)
    plt.title('Fare After Transform')

    plt.show()
    

### ⚠️ Testing `np.sin` Isn't a Meaningful Choice Here

`np.sin` is periodic — it repeats every 2π and folds large, meaningfully-different `Fare` values back on top of each other, destroying information rather than reshaping the distribution sensibly. It runs without error (this cell is really just demonstrating that `apply_transform()` accepts *any* function), but it isn't a transform anyone would actually pick for skewed, always-positive data like `Fare`.

In [ ]:
apply_transform(np.sin)

### More Meaningful Alternatives — `sqrt` and `reciprocal`

For monotonically increasing, always-positive skewed data, `sqrt` (a gentler pull-in than log) and `reciprocal` (a much stronger one) are the other classic choices worth comparing directly against `log1p`.

In [ ]:
apply_transform(np.sqrt)


In [ ]:
apply_transform(lambda x: 1 / (x + 1))   # reciprocal; +1 avoids division by zero


## Summary

| Task | Method |
|---|---|
| Fix missing values safely | `df.fillna({'col': value}, inplace=True)` *(not chained `df['col'].fillna(..., inplace=True)`)* |
| Check normality visually | `sns.histplot(kde=True)` + `stats.probplot(col, dist='norm', plot=plt)` |
| Check normality numerically | `df[col].skew()` — closer to 0 is more symmetric |
| Apply a custom transform | `FunctionTransformer(func=...)` |
| Apply a transform to only some columns | `ColumnTransformer([('name', FunctionTransformer(...), [col])], remainder='passthrough')` |
| Reliable before/after comparison | `cross_val_score(model, X, y, cv=10)` |

### What's actually worth trying, for skewed positive data
`log1p` → `sqrt` → `reciprocal`, in roughly increasing order of how aggressively they pull in a long right tail. The next notebook covers `PowerTransformer` (Box-Cox / Yeo-Johnson), which finds the *best* transform automatically instead of guessing a function by hand.